In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import astropy.units as u
import astropy.visualization
import named_arrays as na
import msfc_ccd

In [ ]:
astropy.visualization.quantity_support();

In [ ]:
fe55 = msfc_ccd.Fe55()

print(f"K-alpha  {fe55.energy_k_alpha:0.4f}  ->  {fe55.charge_k_alpha:0.1f}")
print(f"K-beta   {fe55.energy_k_beta:0.4f}  ->  {fe55.charge_k_beta:0.1f}")

In [ ]:
axis_time = "time"

images = msfc_ccd.fits.open(
    path=na.ScalarArray(
        ndarray=np.array(msfc_ccd.samples.paths_fe55_esis3),
        axes=axis_time,
    ),
)

taps = images.taps

axis_x = taps.axis_x
axis_y = taps.axis_y
axis_tap_x = taps.axis_tap_x
axis_tap_y = taps.axis_tap_y

In [ ]:
hits = taps.hits()

num = np.sum(np.isfinite(hits.outputs), axis=(axis_time, axis_x, axis_y))

print(f"events found in each tap, over {taps.shape[axis_time]} images:")
print(num)

In [ ]:
axis_charge = "charge"

hist = na.histogram(
    a=hits.outputs,
    bins={axis_charge: 81},
    axis=(axis_time, axis_x, axis_y),
    min=0 * u.DN,
    max=800 * u.DN,
)

edges = hist.inputs
centers = (edges[{axis_charge: slice(None, -1)}] + edges[{axis_charge: slice(1, None)}]) / 2

In [ ]:
fig, ax = na.plt.subplots(
    axis_rows=axis_tap_y,
    axis_cols=axis_tap_x,
    nrows=taps.shape[axis_tap_y],
    ncols=taps.shape[axis_tap_x],
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
na.plt.stairs(
    edges,
    hist.outputs,
    axis=axis_charge,
    ax=ax,
    baseline=None,
)
na.plt.set_ylabel("number of events", ax[{axis_tap_x: 0}])
na.plt.set_xlabel("charge (DN)", ax=ax[{axis_tap_y: 0}])
na.plt.text(
    x=0.05,
    y=0.95,
    s=taps.label,
    ax=ax,
    transform=na.plt.transAxes(ax),
    ha="left",
    va="top",
);

In [ ]:
tap = {axis_tap_x: 0, axis_tap_y: 0}
peak = centers[np.argmax(hist.outputs[tap], axis=axis_charge)]

# The event in the first image whose charge is closest to the peak
charge = hits.outputs[tap][{axis_time: 0}]
index = np.nanargmin(np.abs(charge - peak))
index = {ax: index[ax].ndarray for ax in index}

half = 4
region = {
    axis_x: slice(index[axis_x] - half, index[axis_x] + half + 1),
    axis_y: slice(index[axis_y] - half, index[axis_y] + half + 1),
}

fig, ax = plt.subplots(figsize=(4, 3.5), constrained_layout=True)
image = na.plt.pcolormesh(
    C=taps.unbiased.active.outputs[tap][{axis_time: 0}][region].value,
    axis_rgb=None,
    ax=ax,
)
ax.set_xlabel("detector $x$ (pix)")
ax.set_ylabel("detector $y$ (pix)")
plt.colorbar(image.ndarray.item(), ax=ax, label="signal (DN)");

In [ ]:
gain = taps.gain().outputs

gain

In [ ]:
charge_k_alpha = (fe55.charge_k_alpha / gain).to(u.DN)
charge_k_beta = (fe55.charge_k_beta / gain).to(u.DN)

fig, ax = na.plt.subplots(
    axis_rows=axis_tap_y,
    axis_cols=axis_tap_x,
    nrows=taps.shape[axis_tap_y],
    ncols=taps.shape[axis_tap_x],
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
na.plt.stairs(
    edges,
    hist.outputs,
    axis=axis_charge,
    ax=ax,
    baseline=None,
)
na.plt.axvline(
    x=charge_k_alpha,
    ax=ax,
    color="tab:orange",
    linestyle="--",
    label=r"fitted K-$\alpha$",
)
na.plt.axvline(
    x=charge_k_beta,
    ax=ax,
    color="tab:green",
    linestyle="--",
    label=r"fitted K-$\beta$",
)
na.plt.set_xlim(400 * u.DN, 800 * u.DN, ax=ax)
na.plt.set_ylabel("number of events", ax[{axis_tap_x: 0}])
na.plt.set_xlabel("charge (DN)", ax=ax[{axis_tap_y: 0}])
handles, labels = ax.ndarray.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper center", ncols=2);

In [ ]:
gain.reshape({"tap": -1})